# Grade coffee beans with kNN in NumPy

A roaster grades green coffee beans 0, 1 or 2 from two lab measurements: moisture, in
percent, and density, in grams per liter. Density runs in the hundreds and moisture
around ten, so a distance on the raw numbers is decided almost entirely by density.
Here you write the
whole classifier with NumPy: every query-to-training distance in one matrix expression,
standardization with the training rows' statistics, and a vote that settles ties.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)

def beans(n):
    grade = rng.integers(0, 3, n)
    moisture = 10.5 + 0.9 * grade + 0.35 * rng.standard_normal(n)   # percent
    density = 680 + 6 * grade + 30 * rng.standard_normal(n)          # grams per liter
    return np.c_[moisture, density], grade

X_train, y_train = beans(60)
X_test, y_test = beans(30)
print("training rows:", X_train.shape, " test rows:", X_test.shape)
print("first three training rows:")
print(X_train[:3].round(2), y_train[:3])

## Part 1 (coding task): All squared distances at once

Write `sq_dists(A, B)` for an $m \times d$ array `A` and an $n \times d$ array `B`. It
returns the $m \times n$ array whose entry `[i, j]` is the squared Euclidean distance
between row $i$ of `A` and row $j$ of `B`. Use
$\lVert \mathbf{a} - \mathbf{b} \rVert^2 = \lVert \mathbf{a} \rVert^2 - 2\,\mathbf{a}^\top \mathbf{b} + \lVert \mathbf{b} \rVert^2$
with broadcasting and one matrix product, no Python loop. Rounding can leave tiny
negative numbers where a distance is 0: replace anything below 0 with 0.

In [ ]:
def sq_dists(A, B):
    """m x n squared Euclidean distances between the rows of A and the rows of B."""
    # TODO
    raise NotImplementedError

In [ ]:
A = np.array([[0.0, 0.0, 1.0], [2.0, -1.0, 0.0]])
B = np.array([[1.0, 1.0, 1.0], [0.0, 0.0, 1.0], [3.0, 0.0, -2.0], [2.0, -1.0, 0.0]])
D = sq_dists(A, B)
assert np.shape(D) == (2, 4), f"expected shape (2, 4): one row per row of A, one column per row of B; got {np.shape(D)}"
by_loop = np.array([[((a - b) ** 2).sum() for b in B] for a in A])
assert np.allclose(D, by_loop), f"the distances are wrong:\n{D}\nexpected\n{by_loop}"
P = np.random.default_rng(0).normal(size=(50, 3)) * 1e3
same = sq_dists(P, P)
assert np.all(same >= 0), "some squared distances are negative: clip the rounding errors at 0"
assert np.allclose(np.diag(same), 0, atol=1e-6), "the distance from a point to itself should be 0"
print("Part 1 passed")

## Part 2 (coding task): Standardize, find the neighbors, vote

Write `knn_grade(X_ref, y_ref, Q, k)`, where `X_ref` and `y_ref` are the labeled beans and
`Q` holds the beans to grade. It returns an integer array with one grade per row of `Q`:

1. Standardize each feature by the mean and standard deviation of the labeled beans
   `X_ref` (`std` with its default `ddof=0`, as `StandardScaler` does), and move the
   labeled beans and the beans in `Q` by that same transform.
2. For each bean in `Q`, take the $k$ labeled beans nearest to it, by `sq_dists` on the
   standardized data.
3. Predict the grade with the most votes among those $k$. When several grades tie for
   the most votes, predict the tied grade whose own nearest member is closest to the
   bean being graded.

The labels are the integers $0, \dots, C - 1$.

In [ ]:
def knn_grade(X_ref, y_ref, Q, k):
    """Grade each row of Q by a vote of its k nearest labeled beans, after scaling."""
    # TODO
    raise NotImplementedError

In [ ]:
pred = np.asarray(knn_grade(X_train, y_train, X_test, 5))
assert pred.shape == (30,), f"expected one prediction per test row, shape (30,); got {pred.shape}"
assert np.issubdtype(pred.dtype, np.integer), "the predictions should be integer grades"
accuracy = (pred == y_test).mean()
assert accuracy > 0.6, (f"test accuracy {accuracy:.2f}: on raw units density decides every distance; "
                        "standardize both features with the training mean and standard deviation")
alone = np.array([knn_grade(X_train, y_train, X_test[i:i + 1], 5)[0] for i in range(5)])
assert np.array_equal(alone, pred[:5]), ("a query predicted on its own gets a different grade than in the batch: "
                                         "the scaling must come from the training rows only, never from the queries")
expected = [2, 2, 2, 1, 0, 2, 0, 1, 2, 2, 0, 0, 1, 1, 2, 0, 0, 0, 1, 0, 0, 2, 2, 2, 1, 0, 2, 0, 2, 1]
assert np.array_equal(pred, expected), f"some predictions differ from the expected ones:\n{pred.tolist()}\n{expected}"
X_tie = np.array([[1.0, 0.0], [2.0, 1.0], [3.0, 0.0], [4.0, 1.0], [10.0, 0.0]])
y_tie = np.array([2, 0, 2, 0, 1])
tie = knn_grade(X_tie, y_tie, np.array([[0.0, 0.5]]), 4)[0]
assert tie == 2, (f"got grade {tie}: the four nearest vote 2, 0, 2, 0, a tie, and the nearest of them is grade 2. "
                  "np.bincount(...).argmax() settles every tie on the smallest label")
print("Part 2 passed")

## Solution

### Part 1: All squared distances at once

In [ ]:
def sq_dists(A, B):
    """m x n squared Euclidean distances between the rows of A and the rows of B."""
    a2 = (A ** 2).sum(axis=1)[:, None]      # (m, 1)
    b2 = (B ** 2).sum(axis=1)[None, :]      # (1, n)
    return np.maximum(a2 - 2 * A @ B.T + b2, 0.0)

**Why it works.** Expanding $\lVert \mathbf{a} - \mathbf{b} \rVert^2$ gives three terms. `a2` has
shape $(m, 1)$ and `b2` shape $(1, n)$, so adding them broadcasts to $(m, n)$, and
`A @ B.T` is the $(m, n)$ table of dot products. The cost is one matrix product instead
of $mn$ Python iterations. The expansion subtracts large numbers that nearly cancel when
two points coincide, which is why a distance of 0 can come out as $-10^{-10}$, and the
clip removes it.

### Part 2: Standardize, find the neighbors, vote

In [ ]:
def knn_grade(X_ref, y_ref, Q, k):
    """Grade each row of Q by a vote of its k nearest labeled beans, after scaling."""
    mean, std = X_ref.mean(axis=0), X_ref.std(axis=0)
    D = sq_dists((Q - mean) / std, (X_ref - mean) / std)
    nearest = np.argsort(D, axis=1, kind="stable")[:, :k]     # nearest first
    n_classes = y_ref.max() + 1
    out = np.empty(len(Q), dtype=int)
    for r, idx in enumerate(nearest):
        labels = y_ref[idx]                                   # in order of distance
        votes = np.bincount(labels, minlength=n_classes)
        tied = np.flatnonzero(votes == votes.max())
        out[r] = labels[np.isin(labels, tied)][0]               # closest among the tied grades
    return out

**Why it works.** The mean and standard deviation are computed once, from the training rows, and the
queries are moved by the same amounts. That makes a query's standardized coordinates
depend only on the query and the training set, so a query predicted alone gets the same
grade as in a batch. Scaling the queries by their own statistics fails that check, and
a single query has a standard deviation of 0. After scaling, a moisture gap of one
standard deviation counts as much as a density gap of one standard deviation, and the
accuracy rises from about 0.3 on raw units to 0.93. `argsort` lists the neighbors
nearest first, so the first label that belongs to a tied grade is the tied grade with
the closest member.